In [1]:
import pandas as pd
import sqlite3

df=pd.read_excel("week12_transactions.xlsx")

conn = sqlite3.connect("momo.db")

df.to_sql("transactions",conn, if_exists="replace", index=False)

print("dataset loaded successfully!")




dataset loaded successfully!


In [2]:
query ="""

SELECT txn_id,district,amount_rwf FROM transactions WHERE txn_type="send" AND amount_rwf>10000 ORDER BY amount_rwf DESC LIMIT 5;
"""
result =pd.read_sql(query,conn)

print(result)

   txn_id    district  amount_rwf
0  T01708    Kicukiro      140000
1  T00364        Huye      139000
2  T00895     Muhanga      139000
3  T00899  Nyarugenge      139000
4  T01314      Gasabo      139000


In [3]:
query ="""
SELECT COUNT (*) AS flagged_count
FROM transactions WHERE flagged = 1;
"""
result= pd.read_sql(query,conn)
print(result
)

   flagged_count
0             11


In [6]:
query="""
SELECT txn_type, COUNT(*) AS total_transactions, SUM(amount_rwf) AS total_amount ,AVG(amount_rwf) AS average_amount

FROM transactions GROUP BY txn_type ORDER BY total_amount DESC;
"""
result=pd.read_sql(query,conn)
print(result)

print("the transaction type moves the most money\n is send, with a total 62049000RWF. paybill has the highest \naverage transaction amount\n, but send has the highest total amount ")

  txn_type  total_transactions  total_amount  average_amount
0     send                 868      62049000    71485.023041
1  paybill                 330      29970000    90818.181818
2  receive                 489      27024000    55263.803681
3  airtime                 313        833000     2661.341853
the transaction type moves the most money
 is send, with a total 62049000RWF. paybill has the highest 
average transaction amount
, but send has the highest total amount 


In [7]:
query="""
SELECT district, SUM(amount_rwf) AS total_amount FROM transactions
GROUP BY district ORDER BY total_amount DESC ;

"""
result=pd.read_sql(query,conn)
print(result)

     district  total_amount
0     Muhanga      21076000
1      Gasabo      20738000
2        Huye      18736000
3  Nyarugenge      18091000
4    Kicukiro      11338000
5     Musanze      10490500
6      Rubavu      10110500
7   Nyagatare       9296000


In [18]:
print("using sql")
query="""
SELECT hour,COUNT(*) AS total_transactions
FROM transactions GROUP BY hour ORDER BY total_transactions DESC 
"""
result=pd.read_sql(query,conn)
print(result)




using sql
    hour  total_transactions
0     20                 185
1     17                 183
2     18                 176
3     12                 170
4     11                 169
5     10                 168
6      8                 168
7     19                 158
8      9                 144
9     16                  38
10     3                  37
11    15                  35
12    13                  35
13    14                  34
14     4                  34
15     2                  34
16     6                  33
17     5                  33
18     0                  33
19     1                  32
20    21                  26
21    23                  25
22    22                  25
23     7                  25


In [19]:
print("====================pandas====================")
print("\nusing pandas")

hourly_transactions=df.groupby("hour").size()

hourly_transactions =hourly_transactions.sort_values(
  ascending =False
)
print(hourly_transactions)


====================pandas====================

using pandas
hour
20    185
17    183
18    176
12    170
11    169
10    168
8     168
19    158
9     144
16     38
3      37
13     35
15     35
14     34
2      34
4      34
0      33
6      33
5      33
1      32
21     26
7      25
22     25
23     25
dtype: int64


In [21]:
import json

json_data="""
{
"city":"kigali","readings":[24,26,28,25,27]
}
"""
data=json.loads(json_data)
city=data["city"]
readings=data["readings"]
number_of_readings=len(readings)
average = sum(readings)/len(readings)
print("city:",city)
print("number of readings:",number_of_readings)
print("average:",round(average,2))

city: kigali
number of readings: 5
average: 26.0


In [ ]:
import requests

# Kigali coordinates
latitude = -1.9441
longitude = 30.0619

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "hourly": "temperature_2m",
    "forecast_days": 1,
    "timezone": "Africa/Kigali"
}

response = requests.get(url, params=params)

data = response.json()

# Extract temperature and time
temperatures = data["hourly"]["temperature_2m"]
times = data["hourly"]["time"]

# Number of hours
number_of_hours = len(temperatures)

# Maximum and minimum temperature
maximum_temperature = max(temperatures)
minimum_temperature = min(temperatures)

# Find the hottest hour
hottest_index = temperatures.index(maximum_temperature)

hottest_time = times[hottest_index]

print("Number of hours:", number_of_hours)
print("Maximum temperature:", maximum_temperature)
print("Minimum temperature:", minimum_temperature)
print("Hottest hour:", hottest_time)

Number of hours: 24
Maximum temperature: 24.1
Minimum temperature: 16.6
Hottest hour: 2026-11-23T14:00


In [27]:
import pandas as pd

# Create DataFrame
weather_df = pd.DataFrame({
    "time": times,
    "temp_c": temperatures
})

# Display descriptive statistics
print(weather_df.describe())

# Count temperatures greater than or equal to 25
hot_hours = weather_df[weather_df["temp_c"] >= 25]

print("Hours with temperature >= 25°C:", len(hot_hours))

          temp_c
count  24.000000
mean   19.050000
std     2.501304
min    16.600000
25%    16.950000
50%    18.350000
75%    20.250000
max    24.100000
Hours with temperature >= 25°C: 0


In [31]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
  .appName("mobileMoneyAnalysis")\
  .master("local[*]")\
  .getOrCreate()

df.to_csv("week12_transactions.csv",index=False)

sdf=spark.read.csv(
  "week12_transactions.csv",
  header=True,
  inferSchema=True
)

print("total rows:",sdf.count())
sdf.printSchema()
sdf.show(5)

c:\Users\SBS\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


total rows: 2000
root
 |-- txn_id: string (nullable = true)
 |-- date: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)

+------+-----+----+----------+--------+----------+-------+
|txn_id| date|hour|  district|txn_type|amount_rwf|flagged|
+------+-----+----+----------+--------+----------+-------+
|T00001|46327|   8|    Gasabo|    send|      1000|      0|
|T00002|46335|  12|Nyarugenge|    send|     25000|      0|
|T00003|46345|   8|   Muhanga|    send|     51000|      0|
|T00004|46327|  17|      Huye|    send|     79000|      0|
|T00005|46341|  12|      Huye|    send|    109000|      0|
+------+-----+----+----------+--------+----------+-------+
only showing top 5 rows


In [33]:
large_transactions =sdf.filter(
  sdf.amount_rwf > 100000
)

print("transactions above 100,000:",large_transactions.count())

#quesstion b

flagged_transaction = sdf.filter(
  sdf.flagged ==1

)
flagged_transaction.show(5)

transactions above 100,000: 451
+------+-----+----+--------+--------+----------+-------+
|txn_id| date|hour|district|txn_type|amount_rwf|flagged|
+------+-----+----+--------+--------+----------+-------+
|T00279|46355|  19| Muhanga|    send|    127000|      1|
|T00326|46333|  22|    Huye|    send|    127000|      1|
|T00536|46353|  17|  Gasabo|    send|    127000|      1|
|T00655|46356|  17|  Rubavu|    send|    134000|      1|
|T00697|46327|   9| Muhanga|    send|    127000|      1|
+------+-----+----+--------+--------+----------+-------+
only showing top 5 rows


In [35]:
sdf.groupBy("txn_type")\
.sum("amount_rwf")\
.show()

#question 3.3 b

district_totals =sdf.groupBy("district")\
  .sum("amount_rwf")

district_pd=district_totals.toPandas()

district_pd=district_pd.sort_values(
    by="sum(amount_rwf)", ascending=False
  )

print(district_pd)

+--------+---------------+
|txn_type|sum(amount_rwf)|
+--------+---------------+
| paybill|       29970000|
| airtime|         833000|
| receive|       27024000|
|    send|       62049000|
+--------+---------------+



c:\Users\SBS\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\SBS\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


     district  sum(amount_rwf)
4     Muhanga         21076000
5      Gasabo         20738000
6        Huye         18736000
1  Nyarugenge         18091000
0    Kicukiro         11338000
2     Musanze         10490500
7      Rubavu         10110500
3   Nyagatare          9296000


In [38]:
spark_result = spark.sql("""
SELECT
    txn_type,
    COUNT(*) AS total_transactions,
    SUM(amount_rwf) AS total_amount,
    AVG(amount_rwf) AS average_amount
FROM transactions
GROUP BY txn_type
ORDER BY total_amount DESC
""")

spark_result.show()

#It allows developers to reuse SQL knowledge and queries across different data-processing environments. SQLite is suitable for smaller local databases, while Spark can distribute processing across many computers to handle massive datasets.

+--------+------------------+------------+------------------+
|txn_type|total_transactions|total_amount|    average_amount|
+--------+------------------+------------+------------------+
|    send|               868|    62049000| 71485.02304147465|
| paybill|               330|    29970000| 90818.18181818182|
| receive|               489|    27024000|  55263.8036809816|
| airtime|               313|      833000|2661.3418530351437|
+--------+------------------+------------+------------------+



# Bonus question: 

Why is it powerful that the same SQL runs on SQLite and a thousand-machine cluster?

# Answer:

It allows developers to reuse SQL knowledge and queries across different data-processing environments. SQLite is suitable for smaller local databases, while Spark can distribute processing across many computers to handle massive datasets.

 # Part 4 — Choose Your Weapon (3 marks)
1. Ministry of Health provides access to PostgreSQL and requests monthly totals per district.	
  -> SQL	SQL is designed to retrieve, filter and aggregate data directly from relational databases.
2. A group has a 40,000-row CSV requiring cleaning, three charts and a regression model.	
  ->Pandas	Pandas is useful for cleaning, manipulating and preparing datasets for analysis and machine learning.
3. A telecom company wants to analyze 40 TB of nationwide call records already on its cluster.	
  ->Spark	Spark supports distributed processing of very large datasets.
4. A dashboard must always display today's weather without uploading a file.	
  ->API	An API retrieves current weather information directly from an external service.


  # Final Project Proposal: Mobile Money Fraud Detection

Our project aims to investigate whether mobile-money transactions can be classified as potentially suspicious or normal based on their transaction amount, time, district and transaction type. We will use the mobile-money transactions dataset provided for Week 12, containing 2,000 transaction records and seven columns. The project is important because detecting suspicious transactions can help financial institutions reduce fraud, protect customers and improve financial security. We plan to use a classification model because the target variable, `flagged`, contains two possible classes: 0 (not flagged) and 1 (flagged). The model will learn patterns from the available transaction information to identify potentially suspicious transactions.